# VEDU reproducible recommendation experiment
Add the two public Kaggle datasets. Source is embedded and checked by SHA256. CPU is sufficient. Save Version / Run All to retain executed outputs.


In [ ]:
from pathlib import Path
import shutil, hashlib, json, sys
WORK=Path('/kaggle/working/vedu-research')
WORK.mkdir(exist_ok=True)
SOURCE_FILES={'engine.py': '"""Reproducible next-item experiments. External data never masquerades as VEDU logs."""\nimport hashlib\nimport json\nimport math\nimport time\nimport platform\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport sklearn\nfrom scipy.sparse import csr_matrix\nfrom sklearn.neighbors import NearestNeighbors\nfrom torch import nn\n\nVERSION = \'vedu-huspm-bpr-gru-1.1\'\nSOURCES = {\n    \'kaggle_logs\': \'https://www.kaggle.com/datasets/ziya07/student-learning-interaction-logs-dataset\',\n    \'kaggle_recommender\': \'https://www.kaggle.com/datasets/nhondangcode/e-learning-recommender-system-dataset\',\n    \'vedu_real\': \'VEDU PostgreSQL: learning.learning_sequence_logs WHERE data_source=REAL\',\n}\n\ndef load_dataset(name, data_dir, rows=None):\n    root = Path(data_dir)\n    if name == \'vedu_real\':\n        df = pd.DataFrame(rows or [])\n        required = {\'user\', \'item\', \'time\', \'utility\'}\n        if not required.issubset(df.columns):\n            raise ValueError(\'VEDU needs REAL events with user/item/time/utility\')\n        hashes = {\'real_export_sha256\': hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()}\n    else:\n        filename = \'student_learning_interaction_dataset.csv\' if name == \'kaggle_logs\' else \'implicit_ratings_en.csv\'\n        path = root / filename\n        raw = pd.read_csv(path)\n        hashes = {filename: hashlib.sha256(path.read_bytes()).hexdigest()}\n        if name == \'kaggle_logs\':\n            df = raw.rename(columns={\'student_id\':\'user\',\'module_id\':\'item\',\'timestamp\':\'time\'})[[\'user\',\'item\',\'time\']].copy()\n            # Only contemporaneous fields. No next_module_prediction, cumulative scores or labels.\n            df[\'utility\'] = 1 + raw.time_spent_minutes.clip(0, 60) / 10 + raw.quiz_score.clip(0, 100) / 100\n        elif name == \'kaggle_recommender\':\n            df = raw.rename(columns={\'user_id\':\'user\',\'item_id\':\'item\',\'created_at\':\'time\'})[[\'user\',\'item\',\'time\']].copy()\n            df[\'utility\'] = 1.0  # Clicks provide no score/duration: neutral utility, explicit limitation.\n        else:\n            raise ValueError(\'Unknown dataset\')\n    original = len(df)\n    df[\'time\'] = pd.to_datetime(df[\'time\'], errors=\'coerce\', utc=True)\n    df = df.dropna(subset=[\'user\',\'item\',\'time\',\'utility\']).copy()\n    df[\'user\'], df[\'item\'] = df.user.astype(str), df.item.astype(str)\n    df[\'utility\'] = pd.to_numeric(df.utility, errors=\'coerce\').clip(lower=0.01,upper=20)\n    df = df.dropna().sort_values([\'time\',\'user\',\'item\'],kind=\'stable\').reset_index(drop=True)\n    return df, {\'source\':SOURCES[name],\'hashes\':hashes,\'raw_rows\':original,\'clean_rows\':len(df),\n                \'dropped_rows\':original-len(df),\'units\':\'timestamps UTC, durations seconds, utility dimensionless\',\n                \'limitations\': \'Kaggle hosting does not prove real collection; author provenance/license must be confirmed. Naive timestamps are assumed UTC for relative ordering, not verified original timezone. Timestamp ties have deterministic, not observed, ordering.\'}\n\ndef mine_huspm(sequences, min_support=0.02, utility_ratio=0.01, max_length=3, max_candidates=200000):\n    """Exact bounded singleton-itemset HUSPM using prefix projection and SWU pruning.\n    Pattern utility = sum of MAX embedding utility in each containing sequence.\n    Carries every matching end position, not just the first embedding.\n    """\n    n = len(sequences)\n    if not n: return [], {\'total_utility\':0,\'candidates\':0,\'min_utility\':0}\n    totals = [sum(v for _,v in s) for s in sequences]\n    threshold = sum(totals)*utility_ratio\n    minimum = max(1,math.ceil(n*min_support))\n    patterns, visited = [], 0\n    def visit(prefix, projected):\n        nonlocal visited\n        children = {}\n        for sid, ends in projected.items():\n            seq = sequences[sid]\n            # Running best prefix utility before each position; optimal embeddings retained.\n            best = -math.inf\n            for pos,(item,value) in enumerate(seq):\n                if pos-1 in ends: best=max(best,ends[pos-1])\n                if best > -math.inf:\n                    children.setdefault(item,{}).setdefault(sid,{})[pos]=best+value\n        for item,projection in sorted(children.items()):\n            visited+=1\n            if visited>max_candidates: raise ValueError(\'HUSPM candidate limit: raise min_support/min_utility or lower max_length. No partial run is accepted.\')\n            if len(projection)<minimum or sum(totals[sid] for sid in projection)<threshold: continue\n            candidate=prefix+[item]\n            utility=sum(max(ends.values()) for ends in projection.values())\n            if utility>=threshold:\n                patterns.append({\'items\':candidate,\'support_count\':len(projection),\'support\':len(projection)/n,\'utility\':utility,\'utility_ratio\':utility/sum(totals)})\n            if len(candidate)<max_length: visit(candidate,projection)\n    visit([], {sid:{-1:0.0} for sid in range(n)})\n    return sorted(patterns,key=lambda p:(-p[\'utility\'],p[\'items\'])), {\'total_utility\':sum(totals),\'min_utility\':threshold,\'candidates\':visited,\'sequence_count\':n,\'exact_within_max_length\':max_length}\n\nclass BPR(nn.Module):\n    def __init__(self, users, items, dim):\n        super().__init__()\n        self.users=nn.Embedding(users,dim); self.items=nn.Embedding(items,dim)\n        nn.init.normal_(self.users.weight,std=0.05); nn.init.normal_(self.items.weight,std=0.05)\n    def loss(self,u,i,j):\n        a,b,c=self.users(u),self.items(i),self.items(j)\n        return -nn.functional.logsigmoid((a*(b-c)).sum(-1)).mean()+0.0001*(a.square().mean()+b.square().mean()+c.square().mean())\n\nclass NextGRU(nn.Module):\n    def __init__(self,items,dim):\n        super().__init__()\n        self.embedding=nn.Embedding(items+1,dim,padding_idx=0)\n        self.gru=nn.GRU(dim,dim,batch_first=True)\n        self.output=nn.Linear(dim,items)\n    def forward(self,x):\n        _,h=self.gru(self.embedding(x))\n        return self.output(h[-1])\n\ndef normalized(scores):\n    scores=np.asarray(scores,dtype=float)\n    if len(scores)==0: return scores\n    return (scores-scores.min())/(scores.max()-scores.min()+1e-12)\n\ndef huspm_scores(history,patterns,count):\n    scores=np.zeros(count)\n    for pattern in patterns:\n        prefix=pattern[\'items\'][:-1]\n        if not prefix: continue\n        cursor=0\n        for item in history:\n            if cursor<len(prefix) and item==prefix[cursor]: cursor+=1\n        if cursor==len(prefix): scores[pattern[\'items\'][-1]]+=pattern[\'utility_ratio\']*pattern[\'support\']\n    return normalized(scores)\n\ndef metrics(cases,score_fn,item_count,ranks_out=None):\n    result={f\'{metric}@{k}\':0.0 for k in (1,3,5,10) if k<=item_count for metric in (\'Recall\',\'NDCG\',\'MRR\')}\n    coverage=set()\n    for user,history,target in cases:\n        scores=np.asarray(score_fn(user,history),dtype=float)\n        ranking=np.argsort(-scores,kind=\'stable\')\n        coverage.update(ranking[:min(10,item_count)].tolist())\n        rank=int(np.where(ranking==target)[0][0])+1\n        if ranks_out is not None: ranks_out.append(rank)\n        for k in (1,3,5,10):\n            if k>item_count: continue\n            result[f\'Recall@{k}\']+=float(rank<=k)\n            result[f\'NDCG@{k}\']+=(1/math.log2(rank+1) if rank<=k else 0)\n            result[f\'MRR@{k}\']+=(1/rank if rank<=k else 0)\n    for key in result: result[key]=result[key]/len(cases) if cases else None\n    for k in (1,3,5,10):\n        if k>item_count or not cases:continue\n        p=result[f\'Recall@{k}\'];n=len(cases);z=1.96;den=1+z*z/n\n        centre=(p+z*z/(2*n))/den\n        margin=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den\n        result[f\'Recall@{k}_ci95\']=[max(0,centre-margin),min(1,centre+margin)]\n    result.update({\'evaluated_users\':len(cases),\'catalog_coverage@10\':len(coverage)/item_count})\n    return result\n\ndef train(dataset,data_dir,output,config=None,rows=None):\n    config=config or {}; seed=int(config.get(\'seed\',42)); epochs=int(config.get(\'epochs\',8)); dim=int(config.get(\'dimensions\',32))\n    if not 1<=epochs<=50 or not 8<=dim<=128: raise ValueError(\'epochs 1-50; dimensions 8-128\')\n    torch.manual_seed(seed); np.random.seed(seed); torch.set_num_threads(2)\n    rng=np.random.default_rng(seed); started=time.perf_counter()\n    df,manifest=load_dataset(dataset,data_dir,rows)\n    if len(df)<30 or df.user.nunique()<3 or df.item.nunique()<3: raise ValueError(\'Need >=30 REAL events, >=3 users, >=3 items before training; no fake metrics\')\n    protocol=config.get(\'protocol\',\'global_temporal\')\n    first=df.time.iloc[int(len(df)*0.70)]; second=df.time.iloc[int(len(df)*0.85)]\n    excluded_users=0\n    if protocol==\'global_temporal\':\n        train_df=df[df.time<first].copy()\n        targets=[df[(df.time>=first)&(df.time<second)],df[df.time>=second]]\n    elif protocol==\'per_user_temporal\':\n        eligible=df.groupby(\'user\').filter(lambda g:len(g)>=5)\n        excluded_users=df.user.nunique()-eligible.user.nunique()\n        # Keep every row at the held-out timestamp outside training. Timestamp ties\n        # are not evidence of observed ordering.\n        training=[]; validation=[]; testing=[]\n        for _,g in eligible.groupby(\'user\',sort=True):\n            times=g.time.drop_duplicates()\n            if len(times)<3:\n                excluded_users+=1;continue\n            validation.append(g[g.time==times.iloc[-2]].head(1))\n            testing.append(g[g.time==times.iloc[-1]].head(1))\n            training.append(g[g.time<times.iloc[-2]])\n        if not training: raise ValueError(\'No eligible per-user histories\')\n        train_df=pd.concat(training).sort_values([\'time\',\'user\',\'item\'],kind=\'stable\')\n        targets=[pd.concat(validation),pd.concat(testing)]\n    else: raise ValueError(\'Unknown split protocol\')\n    items=sorted(train_df.item.unique()); users=sorted(train_df.user.unique())\n    im={v:i for i,v in enumerate(items)}; um={v:i for i,v in enumerate(users)}\n    if len(items)<3: raise ValueError(\'Training split has <3 items\')\n    train_df[\'i\']=train_df.item.map(im); train_df[\'u\']=train_df.user.map(um)\n    histories={int(u):g.i.tolist() for u,g in train_df.groupby(\'u\')}\n    case_details=[]\n    def cases(targets):\n        found=[]; skipped=0\n        details=[]\n        for user,g in targets.groupby(\'user\',sort=True):\n            target=g.iloc[0]; past=df[(df.user==user)&(df.time<target.time)]\n            history=[im[x] for x in past.item if x in im][-20:]\n            reason=\'unknown_item\' if target[\'item\'] not in im else (\'no_known_history\' if not history else None)\n            details.append({\'user_hash\':hashlib.sha256((dataset+\':\'+user).encode()).hexdigest(), \'target\':target[\'item\'], \'target_time\':target.time.isoformat(),\'history_length\':len(history),\'unsupported_reason\':reason})\n            if reason: skipped+=1;continue\n            found.append((um.get(user,-1),history,im[target[\'item\']]))\n        case_details.append(details)\n        return found,skipped\n    val,val_skip=cases(targets[0]); test,test_skip=cases(targets[1])\n    if not val or not test: raise ValueError(\'Temporal validation/test has no usable cases\')\n    positives={u:set(h) for u,h in histories.items()}\n    utility_triples=[]\n    if dataset==\'kaggle_logs\':\n        # A 5-module catalog is nearly saturated: unseen-negative BPR is undefined for\n        # most learners. Use TRAIN-only pairwise average utility preference instead.\n        for user,g in train_df.groupby(\'u\'):\n            values=g.groupby(\'i\').utility.mean().to_dict()\n            utility_triples.extend((int(user),int(i),int(j)) for i in values for j in values if values[i]>values[j]+1e-9)\n    pairs=np.array([(u,i) for u,seen in positives.items() if len(seen)<len(items) for i in seen],dtype=np.int64)\n    if utility_triples:pairs=np.array([(u,i) for u,i,j in utility_triples],dtype=np.int64)\n    # With a tiny catalog all users may have consumed all modules. Sample next-item contrasts,\n    # not false "unseen" negatives; the task permits review/repeated consumption.\n    contrast=False\n    excluded_saturated_users=sum(len(seen)==len(items) for seen in positives.values())\n    if not len(pairs): pairs=train_df[[\'u\',\'i\']].to_numpy(dtype=np.int64); contrast=True\n    bpr=BPR(len(users),len(items),dim); optimizer=torch.optim.Adam(bpr.parameters(),lr=0.01)\n    bpr_loss=[]; best_val=-1; best_state=None\n    for _ in range(epochs):\n        losses=[]\n        for indices in np.array_split(rng.permutation(len(pairs)),max(1,math.ceil(len(pairs)/512))):\n            batch=pairs[indices]; negatives=[]\n            for offset,(u,i) in enumerate(batch):\n                if utility_triples:\n                    negatives.append(utility_triples[int(indices[offset])][2]);continue\n                if contrast:\n                    j=int(rng.integers(len(items)-1)); j+=j>=i\n                else:\n                    j=int(rng.integers(len(items)))\n                    while j in positives[int(u)]: j=int(rng.integers(len(items)))\n                negatives.append(j)\n            optimizer.zero_grad(); loss=bpr.loss(torch.tensor(batch[:,0]),torch.tensor(batch[:,1]),torch.tensor(negatives));loss.backward();optimizer.step();losses.append(float(loss.detach()))\n        bpr_loss.append(float(np.mean(losses)))\n        with torch.no_grad():\n            emb=bpr.items.weight.numpy(); ue=bpr.users.weight.numpy()\n            value=metrics(val,lambda u,h:emb@(ue[u] if u>=0 else emb[h].mean(0)),len(items))[f\'NDCG@{min(3,len(items))}\']\n        if value>best_val: best_val=value; best_state={k:v.detach().clone() for k,v in bpr.state_dict().items()}\n    bpr.load_state_dict(best_state)\n    samples=[]\n    for h in histories.values():\n        for index in range(1,len(h)):\n            context=[i+1 for i in h[max(0,index-20):index]]\n            samples.append(([0]*(20-len(context))+context,h[index]))\n    if not samples: raise ValueError(\'Need histories of length >=2 for GRU\')\n    x=torch.tensor([s[0] for s in samples]); y=torch.tensor([s[1] for s in samples])\n    gru=NextGRU(len(items),dim); optimizer=torch.optim.Adam(gru.parameters(),lr=0.003)\n    gru_loss=[]; best_val=-1; best_state=None\n    def gru_scores(u,h):\n        context=[i+1 for i in h[-20:]]\n        with torch.no_grad(): return gru(torch.tensor([[0]*(20-len(context))+context]))[0].numpy()\n    for _ in range(epochs):\n        losses=[]\n        for indices in np.array_split(rng.permutation(len(samples)),max(1,math.ceil(len(samples)/256))):\n            optimizer.zero_grad(); loss=nn.functional.cross_entropy(gru(x[indices]),y[indices]);loss.backward();nn.utils.clip_grad_norm_(gru.parameters(),5);optimizer.step();losses.append(float(loss.detach()))\n        gru_loss.append(float(np.mean(losses)))\n        gru.eval();value=metrics(val,gru_scores,len(items))[f\'NDCG@{min(3,len(items))}\']\n        if value>best_val:best_val=value;best_state={k:v.detach().clone() for k,v in gru.state_dict().items()}\n    gru.load_state_dict(best_state);gru.eval()\n    emb=bpr.items.weight.detach().numpy(); ue=bpr.users.weight.detach().numpy()\n    bpr_scores=lambda u,h:emb@(ue[u] if u>=0 else emb[h].mean(0))\n    # External baseline: sklearn cosine NearestNeighbors fitted to training-only item-user vectors.\n    unique=train_df[[\'u\',\'i\']].drop_duplicates()\n    matrix=csr_matrix((np.ones(len(unique)),(unique.i,unique.u)),shape=(len(items),len(users)))\n    neighbor=NearestNeighbors(metric=\'cosine\',algorithm=\'brute\',n_neighbors=min(31,len(items))).fit(matrix)\n    distance,index=neighbor.kneighbors(matrix)\n    similarity=np.zeros((len(items),len(items)),dtype=np.float32)\n    for i in range(len(items)):\n        for d,j in zip(distance[i],index[i]):\n            if i!=j:similarity[i,j]=max(0,1-d)\n    knn_scores=lambda u,h:similarity[h[-20:]].sum(axis=0)\n    sequences=[[(int(row.i),float(row.utility)) for row in g.tail(100).itertuples()] for _,g in train_df.groupby(\'u\')]\n    mining_started=time.perf_counter()\n    patterns,mining=mine_huspm(sequences,float(config.get(\'min_support\',0.02)),float(config.get(\'utility_ratio\',0.01)),int(config.get(\'max_length\',3)))\n    mining[\'runtime_seconds\']=round(time.perf_counter()-mining_started,6)\n    h_scores=lambda u,h:huspm_scores(h,patterns,len(items))\n    def cached(fn):\n        values={}\n        def score(u,h):\n            key=(u,tuple(h))\n            if key not in values: values[key]=fn(u,h)\n            return values[key]\n        return score\n    # Weight search reuses predictions; no repeated GRU inference per weight.\n    bpr_scores=cached(bpr_scores); gru_scores=cached(gru_scores); h_scores=cached(h_scores)\n    weight_options=[(a,1-a,0.0) for a in [0,.25,.5,.75,1]]+[(a,b,round(1-a-b,8)) for a,b in [(0.2,0.6),(0.4,0.4),(0.6,0.2),(0.0,0.8),(0.8,0.0),(0.2,0.4)]]\n    def hybrid(weights):return lambda u,h:weights[0]*normalized(bpr_scores(u,h))+weights[1]*normalized(gru_scores(u,h))+weights[2]*h_scores(u,h)\n    weights=max(weight_options,key=lambda w:metrics(val,hybrid(w),len(items))[f\'NDCG@{min(3,len(items))}\'])\n    pop=train_df.i.value_counts().reindex(range(len(items)),fill_value=0).to_numpy(dtype=float)\n    def without_huspm(a):return lambda u,h:a*normalized(bpr_scores(u,h))+(1-a)*normalized(gru_scores(u,h))\n    no_h_weight=max([0,.25,.5,.75,1],key=lambda a:metrics(val,without_huspm(a),len(items))[f\'NDCG@{min(3,len(items))}\'])\n    scorers={\'BPR-MF\':bpr_scores,\'GRU-next-item\':gru_scores,\'Item-KNN (sklearn external)\':knn_scores,\'Popularity (train-only)\':lambda u,h:pop,\'HUSPM-only\':h_scores,\'BPR+GRU without HUSPM (ablation)\':without_huspm(no_h_weight),\'Hybrid BPR+GRU+HUSPM\':hybrid(weights)}\n    def random_scores(u,h):\n        # Stable for a given seed/context; independent of model RNG and call order.\n        key=hashlib.sha256(json.dumps([seed,u,h]).encode()).digest()\n        return np.random.default_rng(int.from_bytes(key[:8],\'big\')).random(len(items))\n    scorers[\'Random (seeded)\']=random_scores\n    measured={}; rank_rows=[]\n    for name,fn in scorers.items():\n        measured[name]={}\n        for split,cases_,details in [(\'validation\',val,case_details[0]),(\'test\',test,case_details[1])]:\n            ranks=[]; measured[name][split]=metrics(cases_,fn,len(items),ranks)\n            rank_iter=iter(ranks)\n            for detail in details:\n                rank_rows.append({**detail,\'model\':name,\'split\':split,\'rank\':None if detail[\'unsupported_reason\'] else next(rank_iter)})\n    for values in measured.values():\n        total_targets=len(test)+test_skip\n        values[\'test_all_targets\']={key:value*len(test)/total_targets for key,value in values[\'test\'].items() if key.startswith((\'Recall@\',\'MRR@\',\'NDCG@\')) and isinstance(value,(int,float))}\n        values[\'test_all_targets\'].update({\'total_target_users\':total_targets,\'known_catalog_with_history\':len(test),\'unsupported_targets_counted_as_misses\':test_skip})\n    manifest.update({\'dataset\':dataset,\'version\':VERSION,\'code_sha256\':hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\'dependencies\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'torch\':torch.__version__,\'sklearn\':sklearn.__version__},\'seed\':seed,\'train_rows\':len(train_df),\'users\':len(users),\'items\':len(items),\'train_before\':first.isoformat(),\'validation_before\':second.isoformat(),\'test_end\':df.time.max().isoformat(),\'validation_skipped_cold\':val_skip,\'test_skipped_cold\':test_skip,\'protocol\':\'global temporal 70/15/15; one first next-item target/user/split; full training catalog; repeats allowed; checkpoints and blend selected on validation only\',\'bpr_objective\':\'train-only average-utility pairwise preference\' if utility_triples else \'implicit observed versus unseen preference\',\'bpr_contrast_negatives\':contrast,\'bpr_saturated_users\':excluded_saturated_users,\'bpr_training_pairs\':len(pairs),\'sequence_window_huspm\':100,\'context_window_gru\':20,\'utility_formula\':\'1 + min(time_spent_seconds,3600)/600 + quiz_score/100\' if dataset==\'kaggle_logs\' else (\'1 per click\' if dataset==\'kaggle_recommender\' else \'1 + verified_actual_seconds/600; capped at 20\'),\'config\':config,\'hybrid_weights\':weights,\'runtime_seconds\':round(time.perf_counter()-started,3)})\n    manifest[\'ablation_bpr_weight\']=no_h_weight\n    if protocol==\'per_user_temporal\':\n        manifest.update({\'train_before\':None,\'validation_before\':None,\'protocol\':\'per-user temporal; >=5 events; last distinct timestamp test, penultimate validation; full training catalog; repeats allowed; validation-only selection\'})\n    manifest.update({\'split_protocol\':protocol,\'excluded_short_or_tied_users\':int(excluded_users),\'evaluation_note\':\'Per-user split is a warm-history diagnostic and can overlap global periods; global temporal is the deployment-oriented protocol. Repeats allowed; full train catalog; unknown targets retained as misses.\',\'train_split_sha256\':hashlib.sha256(train_df[[\'user\',\'item\',\'time\',\'utility\']].to_csv(index=False).encode()).hexdigest(),\'hybrid_weight_candidates\':weight_options,\'source_revision\':config.get(\'source_revision\')})\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    artifact={\'manifest\':manifest,\'metrics\':measured,\'mining\':mining,\'patterns\':patterns,\'items\':items,\'users\':users,\'bpr_loss\':bpr_loss,\'gru_loss\':gru_loss,\'dimensions\':dim}\n    torch.save({\'bpr\':bpr.state_dict(),\'gru\':gru.state_dict()},output/\'models.pt\')\n    np.save(output/\'knn.npy\',similarity)\n    pd.DataFrame(rank_rows).to_csv(output/\'per_user_ranks.csv\',index=False)\n    (output/\'report.json\').write_text(json.dumps(artifact,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    artifact[\'artifact_sha256\']=hashlib.sha256((output/\'models.pt\').read_bytes()).hexdigest()\n    (output/\'report.json\').write_text(json.dumps(artifact,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    return artifact\n\ndef predict(artifact_dir,history,external_user=None):\n    path=Path(artifact_dir);report=json.loads((path/\'report.json\').read_text(encoding=\'utf-8\'))\n    items=report[\'items\'];mapping={v:i for i,v in enumerate(items)};h=[mapping[str(x)] for x in history if str(x) in mapping][-20:]\n    if not h:return {\'scores\':[],\'reason\':\'No mapped history; use explicit curriculum fallback, not a model claim.\'}\n    model=torch.load(path/\'models.pt\',map_location=\'cpu\',weights_only=True)\n    bpr=BPR(len(report[\'users\']),len(items),report[\'dimensions\']);bpr.load_state_dict(model[\'bpr\']);bpr.eval()\n    gru=NextGRU(len(items),report[\'dimensions\']);gru.load_state_dict(model[\'gru\']);gru.eval()\n    with torch.no_grad():\n        embedding=bpr.items.weight.numpy()\n        u=report[\'users\'].index(str(external_user)) if str(external_user) in report[\'users\'] else -1\n        user_vector=bpr.users.weight.numpy()[u] if u>=0 else embedding[h].mean(0)\n        a=normalized(embedding@user_vector)\n        context=[i+1 for i in h];b=normalized(gru(torch.tensor([[0]*(20-len(h))+context]))[0].numpy())\n    c=huspm_scores(h,report[\'patterns\'],len(items));w=report[\'manifest\'][\'hybrid_weights\'];scores=w[0]*a+w[1]*b+w[2]*c\n    return {\'scores\':[{\'item\':items[i],\'score\':float(scores[i]),\'bpr\':float(a[i]),\'gru\':float(b[i]),\'huspm\':float(c[i])} for i in np.argsort(-scores)],\'reason\':\'Hybrid; BPR fold-in for unknown users; only approved mappings are used\',\'manifest\':report[\'manifest\'],\'model_sha256\':report[\'artifact_sha256\']}\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser();parser.add_argument(\'--dataset\',choices=list(SOURCES),required=True);parser.add_argument(\'--data\',default=\'data\');parser.add_argument(\'--output\',required=True);parser.add_argument(\'--epochs\',type=int,default=8)\n    args=parser.parse_args();result=train(args.dataset,args.data,args.output,{\'epochs\':args.epochs});print(json.dumps({\'manifest\':result[\'manifest\'],\'metrics\':result[\'metrics\'],\'patterns\':len(result[\'patterns\'])},ensure_ascii=False,indent=2))\n', 'benchmark.py': '"""Offline research runner, shared by Kaggle and PyCharm. Never updates the LMS DB."""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport subprocess\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\nfrom engine import train\n\n\ndef fingerprint(data):\n    return {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(data).glob(\'*.csv\'))}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data\', required=True)\n    parser.add_argument(\'--output\', required=True)\n    parser.add_argument(\'--epochs\', type=int, default=20)\n    parser.add_argument(\'--seeds\', nargs=\'+\', type=int, default=[13,21,42,77,101])\n    parser.add_argument(\'--datasets\', nargs=\'+\', default=[\'kaggle_logs\',\'kaggle_recommender\'])\n    parser.add_argument(\'--protocols\', nargs=\'+\', default=[\'global_temporal\',\'per_user_temporal\'])\n    parser.add_argument(\'--environment\', choices=[\'local\',\'kaggle\'], default=\'local\')\n    args = parser.parse_args()\n    output = Path(args.output); output.mkdir(parents=True, exist_ok=True)\n    source = Path(__file__).parent\n    try:\n        revision = subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'], cwd=source, text=True, stderr=subprocess.DEVNULL).strip()\n    except (OSError,subprocess.CalledProcessError):\n        lock=source/\'research-lock.json\'\n        revision=json.loads(lock.read_text())[\'base_commit\'] if lock.exists() else \'source-upload\'\n    lock = {\'base_commit\':revision,\'environment\':args.environment,\'created_at\':datetime.now(timezone.utc).isoformat(),\n            \'python\':platform.python_version(),\'dataset_sha256\':fingerprint(args.data),\n            \'source_sha256\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in source.glob(\'*.py\')},\n            \'arguments\':vars(args),\'device\':\'CPU; engine does not use CUDA\'}\n    (output/\'research-lock.json\').write_text(json.dumps(lock,indent=2),encoding=\'utf-8\')\n    rows=[]\n    for dataset in args.datasets:\n        for protocol in args.protocols:\n            for seed in args.seeds:\n                config={\'epochs\':args.epochs,\'seed\':seed,\'dimensions\':32,\'min_support\':.02,\'utility_ratio\':.01,\'max_length\':3,\'protocol\':protocol,\'source_revision\':revision}\n                dest=output/dataset/protocol/f\'seed-{seed}\'\n                print(f\'TRAIN {dataset} {protocol} seed={seed}\',flush=True)\n                report=train(dataset,args.data,dest,config)\n                for model,metrics in report[\'metrics\'].items():\n                    row={\'dataset\':dataset,\'protocol\':protocol,\'seed\':seed,\'model\':model,\'environment\':args.environment,\n                         \'epochs\':args.epochs,\'items\':report[\'manifest\'][\'items\'],\'patterns\':len(report[\'patterns\']),\n                         \'runtime_seconds\':report[\'manifest\'][\'runtime_seconds\'],\'huspm_weight\':report[\'manifest\'][\'hybrid_weights\'][2]}\n                    row.update(metrics[\'test\'])\n                    row.update({\'all_\'+k:v for k,v in metrics[\'test_all_targets\'].items()})\n                    rows.append(row)\n                frame=pd.DataFrame(rows)\n                frame.to_csv(output/\'model-results.csv\',index=False)\n                summary=frame.groupby([\'dataset\',\'protocol\',\'model\'])[[\'Recall@1\',\'Recall@3\',\'NDCG@3\',\'MRR@3\',\'all_NDCG@3\',\'catalog_coverage@10\']].agg([\'mean\',\'std\',\'count\'])\n                summary.to_csv(output/\'model-summary.csv\')\n                print(f\'DONE {dataset} {protocol} seed={seed}: {report["manifest"]["runtime_seconds"]}s\',flush=True)\n    print(f\'COMPLETE: {len(rows)} model rows, {len(args.datasets)*len(args.protocols)*len(args.seeds)} runs -> {output}\',flush=True)\n\n\nif __name__==\'__main__\': main()\n', 'requirements.txt': 'numpy>=1.26,<3\npandas>=2.2,<3\nscipy>=1.12,<2\nscikit-learn>=1.5,<2\ntorch>=2.5,<3\n', 'research-lock.json': '{"base_commit": "53a3ff8e7bac2b2f5b66b1587aca8c77cec64da9", "source_sha256": {"engine.py": "305591b408db628963648c045cb7f55ec2b8d4878675650928a0499dbbbbdb66", "benchmark.py": "759af5eddd2ecd697260cdb9b09cabc8326dedb3cbd3becd0b9490c910bd009a", "requirements.txt": "d9190e46c737d2033a188ee6885f4f2ac432e08b52fdcbd7dd23cbd29fd1f010"}, "dataset_sha256": {"implicit_ratings_en.csv": "51e2f759745f26ddafc852e62020c21fcb12351346a502a21f5719ee5b667cb2", "items_en.csv": "5861cd72a467ff50db0531098624f9887e920c985b4b8b492ff721f9dc63f862", "student_learning_interaction_dataset.csv": "44008507494f23207b8598f789505d6b6cb9ee6aceac20b79732dd122101cc42"}}'}
for name,content in SOURCE_FILES.items():
    (WORK/name).write_text(content,encoding="utf-8")
lock=json.loads((WORK/'research-lock.json').read_text())
for name,digest in lock['source_sha256'].items():
    assert hashlib.sha256((WORK/name).read_bytes()).hexdigest()==digest
DATA=WORK/'data'; DATA.mkdir(exist_ok=True)
for name,digest in lock['dataset_sha256'].items():
    matches=list(Path('/kaggle/input').rglob(name))
    assert len(matches)==1, (name, matches)
    shutil.copy2(matches[0],DATA/name)
    assert hashlib.sha256((DATA/name).read_bytes()).hexdigest()==digest, 'Dataset version differs: '+name
print('Source revision',lock['base_commit']); print(lock['dataset_sha256'])


In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pip','install','-r',str(WORK/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(WORK/'benchmark.py'),'--data',str(DATA),'--output','/kaggle/working/results','--epochs','20','--environment','kaggle'],check=True)


In [ ]:
import pandas as pd
results=pd.read_csv('/kaggle/working/results/model-results.csv')
assert len(results)==160
display(results.groupby(['dataset','protocol','model'])[['Recall@3','NDCG@3','all_NDCG@3']].agg(['mean','std']))
shutil.make_archive('/kaggle/working/vedu-kaggle-results','zip','/kaggle/working/results')
print('Download vedu-kaggle-results.zip and Save Version with executed outputs.')
